# Inventory Analysis

## Overview

...

## Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

## 1. Import Data

### 1a. Define data paths

Use the imported Path function to create relative paths to data files.

In [2]:
DATA_FOLDER_PATH = Path("../data")
DATA_1_PATH = DATA_FOLDER_PATH / "diapers.csv"
DATA_2_PATH = DATA_FOLDER_PATH / "form.csv"
DATA_3_PATH = DATA_FOLDER_PATH / "form_2_new.csv"
OUTPUT_DATA_PATH = DATA_FOLDER_PATH / "output/diapers_final.csv"

### 1b. Import data from CSV files

Use pandas to load csv files as DataFrames.

In [3]:
df1 = pd.read_csv(DATA_1_PATH)
df2 = pd.read_csv(DATA_2_PATH, header=1)
df3 = pd.read_csv(DATA_3_PATH)

## 2. Data Cleaning

Because the Google Form used for data collection changed a few times, there are 3 different csv files with varying columns that need to be combined into one csv file.

The output csv file should have the following columns:  
- timestamp (date)
- branch (string)
- zip (string)
- size (string)
- packs (int)

### 2a. File 1 - diapers.csv

Display summary information.

In [4]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 59425 entries, 0 to 59424
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Timestamp  55946 non-null  str    
 1   BRANCH     55930 non-null  str    
 2   ZIP CODE   55938 non-null  str    
 3   SIZE       52387 non-null  str    
 4   # PACKS    55793 non-null  float64
 5   SIZE.1     8397 non-null   str    
 6   # PACKS.1  9265 non-null   float64
 7   SIZE.2     1785 non-null   str    
 8   # PACKS.2  2873 non-null   float64
 9   TOTAL      59425 non-null  int64  
 10  DATE       59425 non-null  str    
 11  MONTH      55946 non-null  str    
 12  YEAR       55946 non-null  str    
 13  DAY        55946 non-null  str    
 14  WEEKDAY    55946 non-null  str    
dtypes: float64(3), int64(1), str(11)
memory usage: 6.8 MB


Select required columns.

In [5]:
df1_filtered: pd.DataFrame = df1.iloc[:, 0:9]

Stack rows.

In [6]:
def stack_rows(df_filtered: pd.DataFrame) -> list[list]:
    df_list = df_filtered.values.tolist()
    combined_data = []
    for record in df_list:
        for i in range(3, 9, 2):
            if record[i] != record[i] and record[i+1] != record[i+1]:
                continue
            row = [*record[0:3], *record[i:i+2]]
            combined_data.append(row)
    return combined_data

In [7]:
combined_data_1: list[list] = stack_rows(df1_filtered)

Create DataFrame.

In [8]:
def create_df(combined_data: list[list]) -> pd.DataFrame:
    columns: list = ["timestamp", "branch", "zip", "size", "packs"]
    df_final: pd.DataFrame = pd.DataFrame(combined_data, columns=columns)
    return df_final

In [9]:
df1_final: pd.DataFrame = create_df(combined_data_1)

### 2b. File 2 - form.csv

Display summary information.

In [10]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 24514 entries, 0 to 24513
Data columns (total 15 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Timestamp                                 24514 non-null  str    
 1   BRANCH                                    24514 non-null  str    
 2   ZIP CODE                                  24514 non-null  str    
 3   What are you submitting stats for?        24514 non-null  str    
 4   Size                                      19367 non-null  str    
 5   Unnamed: 5                                19367 non-null  str    
 6   Number of PACKS OF DIAPERS distributed    19367 non-null  float64
 7   Size.1                                    7267 non-null   str    
 8   Unnamed: 8                                7267 non-null   str    
 9   Number of PACKS OF DIAPERS distributed.1  7129 non-null   float64
 10  Size.2                                    156

Filter to select diaper stats only.

In [11]:
df2_diapers = df2[df2.iloc[:, 3] == "Diapers"]

Select required columns.

In [12]:
df2_col_idx = [0,1,2,4,6,7,9,10,12]
df2_filtered = df2_diapers.iloc[1:,df2_col_idx]

Stack rows.

In [13]:
combined_data_2: list[list] = stack_rows(df2_filtered)

Create DataFrame.

In [14]:
df2_final = create_df(combined_data_2)
df2_final.head()

,timestamp,branch,zip,size,packs
0,1/2/2024 9:29:02,LC,63137,5,1.0
1,1/2/2024 9:38:49,LC,63136,3,1.0
2,1/2/2024 9:38:49,LC,63136,4,1.0
3,1/2/2024 9:48:33,WR,63123,4T,1.0
4,1/2/2024 10:00:31,LC,62040,4,1.0


### 2c. File 3 - form2.csv

Display summary information.

In [15]:
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 8853 entries, 0 to 8852
Data columns (total 13 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   Timestamp                                                        8853 non-null   str    
 1   BRANCH                                                           8827 non-null   str    
 2   ZIP CODE                                                         8853 non-null   str    
 3   Size                                                             7490 non-null   str    
 4   Number of PACKS OF DIAPERS distributed                           7276 non-null   float64
 5   Size.1                                                           2208 non-null   str    
 6   Number of PACKS OF DIAPERS distributed.1                         2144 non-null   float64
 7   Size.2                                               

Select required columns.

In [16]:
df3_filtered = df3.iloc[1:, 0:9]

Stack rows.

In [17]:
combined_data_3: list[list] = stack_rows(df3_filtered)

Create DataFrame.

In [18]:
df3_final = create_df(combined_data_3)

### 2d. Combining DataFrames into One

Concatenating all three DataFrames.

In [19]:
df_combined = pd.concat([df1_final, df2_final, df3_final], axis=0, ignore_index=True)

### 2e. Cleaning by Column

Display counts of na values.

In [20]:
df_combined.isna().sum()

timestamp       1
branch         41
zip            10
size         6647
packs         825
dtype: int64

#### I. timestamp

Find missing row in timestamp column.

In [21]:
df_combined[df_combined["timestamp"].isna()]

,timestamp,branch,zip,size,packs
29164,NaN,----,NaN,0,0.0


Drop row with na timestamp.

In [22]:
df_combined.drop([29164], inplace=True)

Convert column to timestamp.

In [23]:
df_combined["timestamp"] = pd.to_datetime(df_combined["timestamp"], errors='coerce')

Add columns for year and month.

In [24]:
df_combined['year'] = df_combined['timestamp'].dt.year
df_combined['month'] = df_combined['timestamp'].dt.month

#### II. branch

Find rows with missing branch.

In [25]:
df_combined[df_combined["branch"].isna()].head()

,timestamp,branch,zip,size,packs,year,month
59607,2026-05-28 10:41:26,NaN,63136,2,1.0,2026.0,5.0
59642,2026-05-27 10:11:53,NaN,63133,6,1.0,2026.0,5.0
59933,2026-05-19 08:04:41,NaN,63122,NaN,1.0,2026.0,5.0
60069,2026-05-14 11:02:06,NaN,63031,5,1.0,2026.0,5.0
60663,2026-05-04 08:58:41,NaN,63033,6,1.0,2026.0,5.0


Select rows without missing branch.

In [26]:
df_combined = df_combined[~df_combined["branch"].isna()]

#### III. size

Get unique sizes.

In [27]:
df_combined["size"].unique()

<StringArray>
[    '1',    '3T',     '5',     '6',     '3',     'N',    '4T',     '4',
     '2',    '4t',    '2T',    '3t',    '2t',     'n',     '7', 'adult',
     'P',     nan,     '0']
Length: 19, dtype: str

Capitalize all strings in the `size` column.

In [28]:
df_combined["size"] = df_combined["size"].str.upper()

In [29]:
df_combined["size"].unique()

<StringArray>
[    '1',    '3T',     '5',     '6',     '3',     'N',    '4T',     '4',
     '2',    '2T',     '7', 'ADULT',     'P',     nan,     '0']
Length: 15, dtype: str

Size 0 should be N.

In [30]:
df_combined["size"] = df_combined["size"].replace({'0': 'N'})

Count of missing sizes.

In [31]:
df_combined["size"].isna().sum()

np.int64(6634)

I plan to use ratios to guess missing sizes aggregated over longer time periods. I will leave rows with missing sizes alone for now.

#### IV. packs

Pivot table of count of packs by month/year

From the following pivot, we can see that the diaper bank likely started giving out 1 pack of diapers per child instead of at most 2 in September 2022.

In [32]:
pd.set_option('display.max_rows', None)

In [33]:
prepare_for_pivot = df_combined[["packs", "year", "month"]]

In [34]:
pd.pivot_table(prepare_for_pivot, values='packs', index=['year', 'month'], columns=['packs'], aggfunc="sum")

packs         0.0     1.0     2.0    3.0    4.0   5.0    6.0   7.0   8.0   \
year   month                                                                
1902.0 10.0    NaN     NaN     2.0    NaN    NaN   NaN    NaN   NaN   NaN   
1931.0 9.0     NaN     NaN     2.0    NaN    NaN   NaN    NaN   NaN   NaN   
2021.0 9.0     NaN     3.0   384.0  132.0   16.0   NaN    6.0   NaN   NaN   
       10.0    NaN    13.0   548.0  216.0   32.0   NaN    6.0   NaN   NaN   
       11.0    NaN     5.0   638.0  207.0   32.0   NaN   12.0   NaN   NaN   
       12.0    NaN     7.0   712.0  204.0   76.0   NaN   36.0   NaN   8.0   
2022.0 1.0     NaN    24.0   972.0  285.0   72.0  10.0   30.0   NaN   8.0   
       2.0     NaN    31.0   944.0  231.0  132.0   5.0   66.0   NaN   NaN   
       3.0     NaN    14.0  1160.0  366.0  156.0   5.0   42.0   NaN   8.0   
       4.0     NaN    18.0  1298.0  372.0  168.0   5.0   36.0   NaN  32.0   
       5.0     NaN    19.0  1384.0  402.0  236.0   5.0  120.0   7.0  16.0   
       6.0     NaN    12.0  1580.0  423.0  324.0  10.0  126.0   NaN  16.0   
       7.0     NaN    32.0  1936.0  546.0  424.0   5.0   96.0   NaN  24.0   
       8.0     NaN   395.0  1436.0  408.0  272.0   NaN  126.0   NaN   NaN   
       9.0     NaN  1335.0   196.0   39.0   24.0   5.0   18.0   NaN   NaN   
       10.0    NaN  1115.0   226.0   48.0   12.0   NaN    6.0  14.0   NaN   
       11.0    NaN  1090.0   286.0   60.0   16.0   NaN    6.0   NaN   NaN   
       12.0    NaN   945.0   208.0   42.0   16.0  20.0   12.0   7.0   NaN   
2023.0 1.0     NaN  1002.0   282.0   51.0   40.0   NaN    NaN   NaN   NaN   
       2.0     NaN   861.0   228.0   57.0   32.0   5.0   18.0   NaN   NaN   
       3.0     NaN  1018.0   286.0   48.0   16.0   NaN   12.0   NaN   NaN   
       4.0     NaN  1105.0   236.0   81.0   20.0   5.0   12.0   7.0   NaN   
       5.0     NaN  1214.0   306.0   42.0   20.0   5.0    6.0   NaN   NaN   
       6.0     NaN  1163.0   280.0   54.0   28.0   5.0    NaN   NaN   8.0   
       7.0     NaN  1203.0   266.0   75.0   24.0   NaN   12.0   NaN   8.0   
       8.0     NaN  1144.0   312.0   72.0   28.0  10.0   12.0   NaN   NaN   
       9.0     0.0  1049.0   266.0   45.0   28.0   NaN    NaN   NaN   NaN   
       10.0    NaN  1195.0   320.0   60.0   20.0  10.0    NaN   NaN   NaN   
       11.0    NaN  1164.0   256.0   69.0   28.0   5.0    6.0   NaN   NaN   
       12.0    NaN  1055.0   240.0   45.0   28.0  10.0   18.0  14.0   NaN   
2024.0 1.0     NaN  2338.0   296.0   42.0   32.0  20.0    NaN   NaN   NaN   
       2.0     0.0  2362.0   264.0   60.0   24.0   NaN   12.0   NaN   NaN   
       3.0     NaN  2266.0   256.0   66.0   16.0   NaN   24.0   NaN   NaN   
       4.0     NaN  2420.0   296.0   60.0   24.0  10.0    NaN   NaN   NaN   
       5.0     NaN  2834.0   284.0   54.0   24.0  20.0    NaN   NaN   NaN   
       6.0     NaN  2552.0   272.0   84.0   16.0   NaN   24.0   NaN   NaN   
       7.0     NaN  2860.0   292.0   60.0   16.0  20.0   12.0   NaN   NaN   
       8.0     NaN  2876.0   328.0   54.0   16.0  30.0   12.0   NaN   NaN   
       9.0     NaN  2766.0   256.0   54.0   16.0  10.0   24.0  14.0   NaN   
       10.0    NaN  2648.0   284.0   96.0   32.0  20.0   12.0   NaN   NaN   
       11.0    0.0  2400.0   240.0   42.0    NaN  20.0    NaN   NaN   NaN   
       12.0    NaN  2400.0   216.0   60.0   16.0  10.0   12.0   NaN   NaN   
2025.0 1.0     NaN  2436.0   252.0   42.0   48.0   NaN   24.0   NaN   NaN   
       2.0     NaN  2284.0   268.0   48.0   16.0   NaN   12.0   NaN   NaN   
       3.0     NaN  2368.0   224.0   78.0   24.0  20.0   12.0   NaN   NaN   
       4.0     NaN  2548.0   220.0   42.0   16.0  20.0   12.0   NaN   NaN   
       5.0     NaN  2422.0   272.0   78.0   16.0   NaN   24.0   NaN   NaN   
       6.0     NaN  2234.0   252.0   36.0    8.0  30.0   24.0   NaN   NaN   
       7.0     NaN  2428.0   264.0   66.0   40.0  30.0   72.0   NaN   NaN   
       8.0     NaN  2494.0   244.0   90.0   16.0  20.0   24.0   NaN   NaN

I'll go ahead and just use data from January 2023 forwards. Using data from when 2 packs were given out per child could skew analytics and predictive modeling. I could try normalizing it to the current policy (dividing packs given out previous to Sept. 2022 by 2), but this assumes that distribution trends are the same before and after the policy change. Also, 2 were not always given out per child. Sometimes 1 pack was given, sometimes 3 packs were given for 2 childen, etc. It would likely be a messy conversion. 3+ years of data from after the policy change is enough for analytics model building purposes.

Also, the data is incomplete for July 2026, so I'll have the data set end after June 2026.

In [35]:
df_sorted = df_combined[(df_combined["timestamp"] >= "1/1/2023") & (df_combined["timestamp"] < "7/1/2026")].sort_values(by=["timestamp"])

### 2f. Filtering to target branch

Filtering to WR branch.

In [36]:
df_filtered_wr = df_sorted[df_sorted["branch"] == "WR"]

### 2g. Cleaning up DataFrame

Reset index at the end.

In [37]:
df_filtered_wr.reset_index(inplace=True, drop=True)

Set df_filtered_wr to wr for easy reference.

In [38]:
wr = df_filtered_wr

We don't distribute sizes 7 and P at WR, so replace 7 and P with NaN.

In [39]:
wr["size"] = wr["size"].replace({"7": np.nan, "P": np.nan})

We only gave out 1 pack of `ADULT`, and we no longer receive them, so remove that row.

In [40]:
wr = wr[wr["size"] != "ADULT"]

I want to perform analytics on 2023 through 2025 data and see if it leads to correct predictions for 2026.

In [41]:
wr_train = wr[wr["year"] < 2026]
wr_test = wr[wr["year"] >= 2026]

## 3. Exploratory Analysis

In [42]:
wr_train.head()

,timestamp,branch,zip,size,packs,year,month
0,2023-01-03,WR,63123,3,1.0,2023.0,1.0
1,2023-01-03,WR,63125,3T,1.0,2023.0,1.0
2,2023-01-03,WR,63123,4,1.0,2023.0,1.0
3,2023-01-03,WR,63129,4,1.0,2023.0,1.0
4,2023-01-03,WR,63123,4,1.0,2023.0,1.0


In [43]:
wr_train.tail()

,timestamp,branch,zip,size,packs,year,month
16129,2025-12-31 16:04:55,WR,63129,6,2.0,2025.0,12.0
16130,2025-12-31 16:17:49,WR,63123,NaN,1.0,2025.0,12.0
16131,2025-12-31 16:17:49,WR,63123,2T,1.0,2025.0,12.0
16132,2025-12-31 16:43:23,WR,63123,4,1.0,2025.0,12.0
16133,2025-12-31 16:43:23,WR,63123,NaN,1.0,2025.0,12.0


### 3a. Diaper Size Analysis

Size totals by month.

In [44]:
pd.pivot_table(wr_train, values='packs', index=['year', 'month'], columns=['size'], aggfunc="sum")

size             1     2    2T      3    3T      4    4T      5      6     N
year   month                                                                
2023.0 1.0     8.0  17.0   5.0   36.0   2.0   61.0   1.0   66.0  100.0   3.0
       2.0     6.0  11.0   5.0   24.0   4.0   43.0   NaN   79.0   79.0   4.0
       3.0    11.0  15.0   5.0   42.0   2.0   48.0   5.0   61.0   92.0   6.0
       4.0     9.0  13.0   5.0   33.0   6.0   35.0   NaN   66.0   83.0   8.0
       5.0    12.0  11.0   2.0   31.0  12.0   54.0   NaN   58.0  108.0  11.0
       6.0     6.0  13.0   8.0   24.0  14.0   69.0   NaN   53.0   84.0   6.0
       7.0    14.0  16.0   8.0   45.0  13.0   46.0   NaN   66.0  104.0   8.0
       8.0    17.0  13.0   4.0   43.0  16.0   68.0   NaN   72.0  119.0   5.0
       9.0    14.0  13.0   5.0   46.0  12.0   66.0   1.0   62.0  106.0   6.0
       10.0   13.0  17.0   3.0   45.0   9.0   70.0   NaN   60.0  105.0   3.0
       11.0   16.0  16.0   2.0   39.0   6.0   81.0   3.0   59.0  100.0   6.0
       12.0   13.0  13.0   1.0   39.0   3.0   63.0   4.0   89.0   86.0   4.0
2024.0 1.0    28.0  24.0   8.0   64.0  14.0  128.0  18.0  168.0  146.0   2.0
       2.0    28.0  36.0   4.0   66.0  10.0  154.0   6.0  122.0  174.0   6.0
       3.0    20.0  22.0   8.0   58.0  14.0  138.0   8.0  108.0  206.0  22.0
       4.0    24.0  36.0  12.0   76.0  16.0  124.0  12.0   98.0  166.0   8.0
       5.0    20.0  14.0   8.0   62.0   8.0  132.0  20.0  158.0  188.0   6.0
       6.0    22.0  22.0  12.0   64.0  20.0  126.0  12.0  144.0  200.0   4.0
       7.0    36.0  22.0  14.0   52.0  10.0  130.0   8.0  114.0  216.0  14.0
       8.0    20.0  24.0  18.0  100.0  18.0  132.0  20.0  174.0  180.0   6.0
       9.0    16.0  12.0  16.0   64.0  20.0   80.0  18.0  152.0  234.0   6.0
       10.0   18.0  22.0  12.0   40.0  18.0   64.0  26.0  152.0  192.0   6.0
       11.0   20.0  16.0  12.0   52.0   8.0   44.0  18.0  130.0  168.0  10.0
       12.0   26.0  18.0  16.0   32.0  20.0   82.0  20.0  204.0  100.0   6.0
2025.0 1.0    12.0  20.0  18.0   42.0  20.0   98.0  20.0   68.0  194.0  10.0
       2.0    12.0  18.0   6.0   38.0  10.0   88.0  16.0  112.0  160.0  16.0
       3.0    18.0  16.0   2.0   52.0   8.0   58.0  18.0  120.0  172.0   8.0
       4.0    10.0  16.0   8.0   48.0   8.0   84.0  20.0  106.0  222.0  10.0
       5.0    20.0  28.0   4.0   32.0  22.0   58.0  10.0  128.0  222.0   4.0
       6.0    18.0  16.0   8.0   44.0  14.0   64.0  22.0  140.0  176.0  12.0
       7.0    36.0  30.0   6.0   40.0  20.0   48.0  22.0  150.0  184.0  14.0
       8.0    30.0  24.0   8.0   44.0  16.0   64.0  26.0  118.0  228.0  12.0
       9.0    16.0  28.0   8.0   48.0  26.0   92.0  34.0  136.0  170.0  10.0
       10.0   27.0  27.0  12.0   44.0  24.0   61.0  21.0  106.0  169.0   5.0
       11.0    6.0  10.0   4.0   23.0  11.0   29.0   9.0   42.0   95.0   5.0
       12.0    8.0  14.0   4.0   23.0  11.0   43.0  12.0   55.0   88.0   2.0

Main Idea: Test if size ratios are consistent with Chi-Square

### 3b. Diaper Packs Analysis

Main idea: I suspect seasonality. Graph each month for each year on the same graph.

## 4. Predictive Modeling

### 4a. Estimating Diaper Totals

Main idea: Since there is a seasonal trand, use Holt-Winters Exponential Smoothing to predict totals.

### 4b. Estimating Diaper Size Totals

Main idea: number of size N diapers distributed in a month = size N proportion * estimated monthly total

## 5. Evaluating Models

### 5a. Diaper Totals Predictions

Main idea: Find the error in the model's predictions. Compare to 2026 data so far, or maybe validate on 2025 data.

### 5b. Diaper Size Totals Predictions

Main idea: Find the error in the model's predictions. Compare to 2026 data so far, or maybe validate on 2025 data.

## 6. Conclusions

Answer the following questions:
- What did I find out in exploratory analysis?
- How well did the models predict the test set?